In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l7.6 Data quality and eval card

> 🎯 **Goal:** Internalize that clean data beats big data, and learn to ship an
> honest report card alongside every adapted model.

**Why this matters.** This lesson closes the unit with the two things that decide
whether post-training actually works in practice: the quality of the data you train
on, and the honesty of how you report the result. Every technique in u7 (SFT, LoRA,
QLoRA, DPO) is only as good as the examples you feed it and only as trustworthy as
the evaluation you publish.

**The intuition.** A few thousand clean, diverse, well-labeled examples beat a
million noisy ones, because the model faithfully learns whatever you show it,
including the mistakes. Garbage in, garbage out, but confidently. And once you have
a result, an honest report card matters as much as the number itself: a perplexity
of 7 means nothing until you know what baseline it beat, what data produced it, and
where the model still fails.

**The mechanics.** An **eval card** is a small, structured record that travels with
the model. It states: what the model is, the task, the metrics *with a baseline to
compare against* (here, the uniform-guess baseline equal to the vocabulary size),
the data provenance (what corpus, what license, what slice), and the limitations
(what it is not good for). Below we train a small model, measure its perplexity, and
assemble exactly such a card as a plain dictionary you could save as JSON.

In [ ]:
import json
from pocketlm import (CharTokenizer, PocketLM, PocketLMConfig, init_weights,
                      make_batch, estimate_loss, perplexity)

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)
torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=tok.vocab_size, block_size=32, n_layer=2,
                     n_head=2, n_embd=64)
model = init_weights(PocketLM(cfg))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
g = torch.Generator().manual_seed(0)
for _ in range(150 if os.environ.get("POCKETLM_CI") else 400):
    x, y = make_batch(data, 32, 16, generator=g)
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
ppl = perplexity(estimate_loss(model, data, 32, 16, iters=10, generator=g))
eval_card = {
    "model": "PocketLM (u7 demo)",
    "task": "char-level next-token on Gutenberg #100",
    "metrics": {"perplexity": round(ppl, 2), "uniform_baseline": tok.vocab_size},
    "data": "Project Gutenberg eBook #100, public domain, leading 1MB slice",
    "limitations": ["char-level", "tiny model", "demo-scale post-training only"],
}
print(json.dumps(eval_card, indent=2))

**What just happened.** We printed a complete eval card. The perplexity came out
well below the `uniform_baseline` (the vocab size, what a model that guessed
uniformly at random would score), so we can honestly say the model learned
something. Crucially, the card carries that baseline, the data provenance, and the
limitations right next to the number, which is what makes the result reproducible
and trustworthy instead of a bare statistic someone could misread.

⚠️ **Common pitfalls**
- Reporting a metric with no baseline. A perplexity number alone is meaningless,
  always say what it beats.
- Omitting data provenance. If you cannot say where the data came from and under
  what license, the result is neither reproducible nor safe to ship.
- Hiding limitations. An honest card names what the model is bad at, char-level,
  tiny, demo-scale here. Silence about weaknesses is how models get misused.
- Chasing data volume over quality. More noisy examples can make a model worse,
  curate before you scale.

🏋️ **Try it yourself.** Add a `train_examples` count and a `date` field to the card,
then deliberately corrupt part of the corpus (shuffle some characters) before
training and rerun. Watch the perplexity rise and notice how the card now tells an
honest story about *why* the result got worse, that is the eval card earning its
keep.

In [ ]:
assert set(eval_card) >= {"model", "task", "metrics", "data", "limitations"}
assert eval_card["metrics"]["perplexity"] < tok.vocab_size